# 04 - Forecasting de Series Temporales

## Pregunta de negocio: *"¿Podemos predecir costos futuros de combustible?"*

---

## Contexto y Teoría

El **forecasting de series temporales** es una de las herramientas más valiosas en ciencia de datos aplicada a la gestión de flotas. Si podemos predecir el consumo futuro de combustible con precisión razonable, podemos:

- **Optimizar presupuestos** de operación
- **Negociar contratos** de suministro de combustible/electricidad
- **Planificar rutas** más eficientes
- **Detectar anomalías** cuando el consumo real se desvía significativamente del pronóstico

### Modelo ARIMA (AutoRegressive Integrated Moving Average)

ARIMA combina tres componentes:

1. **AR (AutoRegressive, p):** El valor actual depende linealmente de sus valores pasados.
   - $y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + ... + \phi_p y_{t-p} + \epsilon_t$
   - El parámetro `p` indica cuántos rezagos (lags) usar.

2. **I (Integrated, d):** Diferenciación para hacer la serie estacionaria.
   - Si la serie tiene tendencia, la diferenciamos: $y'_t = y_t - y_{t-1}$
   - El parámetro `d` indica cuántas veces diferenciar (usualmente 0 o 1).

3. **MA (Moving Average, q):** El valor actual depende de errores de predicción pasados.
   - $y_t = c + \epsilon_t + \theta_1 \epsilon_{t-1} + ... + \theta_q \epsilon_{t-q}$
   - El parámetro `q` indica cuántos errores pasados usar.

### Selección de parámetros con ACF y PACF

- **ACF (Autocorrelation Function):** Mide la correlación entre $y_t$ y $y_{t-k}$ para distintos rezagos $k$. Un corte abrupto en ACF sugiere el valor de `q`.
- **PACF (Partial Autocorrelation Function):** Mide la correlación directa entre $y_t$ y $y_{t-k}$, eliminando la influencia de los rezagos intermedios. Un corte abrupto en PACF sugiere el valor de `p`.

### Train/Test Split en Series Temporales

**¡NUNCA se hace split aleatorio en series temporales!** El orden cronológico importa. Siempre:
- **Train:** primeros N% de los datos (80% típicamente)
- **Test:** últimos (100-N)% de los datos

Esto simula la realidad: entrenamos con datos históricos y evaluamos con datos futuros.

### Métricas de evaluación

- **RMSE (Root Mean Squared Error):** $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ — Penaliza errores grandes.
- **MAPE (Mean Absolute Percentage Error):** $\frac{100}{n}\sum\left|\frac{y_i - \hat{y}_i}{y_i}\right|$ — Fácil de interpretar (% de error).

### Baselines simples

Siempre comparar modelos complejos contra baselines:
- **Naive:** El valor de mañana = valor de hoy.
- **Seasonal naive:** El valor de mañana = valor del mismo día de la semana pasada.
- **Moving average:** El valor de mañana = promedio de los últimos N días.

---

## 1. Carga de datos y preparación de la serie temporal

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

vtype_colors = {
    'electrico': '#2ecc71',
    'gasolina': '#3498db',
    'hibrido': '#9b59b6',
    'deportivo': '#e74c3c'
}

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
print(f"Raíz del proyecto: {project_root}")

In [ ]:
# Cargar todos los archivos de telemetría
telemetry_files = sorted(glob.glob(os.path.join(project_root, 'data/raw/telemetry/telemetry_*.csv')))
print(f"Archivos de telemetría encontrados: {len(telemetry_files)}")

df_list = []
for f in telemetry_files:
    df_list.append(pd.read_csv(f, parse_dates=['timestamp']))
telemetry = pd.concat(df_list, ignore_index=True)

# Cargar perfiles de flota
fleet = pd.read_csv(os.path.join(project_root, 'data/raw/fleet_profiles.csv'))
telemetry = telemetry.merge(fleet[['vehicle_id', 'vehicle_type']], on='vehicle_id', how='left')

print(f"Registros de telemetría: {len(telemetry):,}")
print(f"Rango temporal: {telemetry['timestamp'].min()} a {telemetry['timestamp'].max()}")
print(f"Vehículos únicos: {telemetry['vehicle_id'].nunique()}")
telemetry.head()

In [ ]:
# Agregar telemetría a consumo diario de la flota
telemetry['date'] = telemetry['timestamp'].dt.date

daily_consumption = (
    telemetry
    .groupby('date')['fuel_consumption_rate']
    .mean()
    .reset_index()
)
daily_consumption.columns = ['date', 'consumo_medio']
daily_consumption['date'] = pd.to_datetime(daily_consumption['date'])
daily_consumption = daily_consumption.set_index('date').sort_index()

# Asegurar frecuencia diaria (rellenar días faltantes si los hay)
daily_consumption = daily_consumption.asfreq('D')
if daily_consumption['consumo_medio'].isna().any():
    n_missing = daily_consumption['consumo_medio'].isna().sum()
    print(f"Días sin datos: {n_missing}. Interpolando...")
    daily_consumption['consumo_medio'] = daily_consumption['consumo_medio'].interpolate(method='linear')

print(f"Serie temporal diaria: {len(daily_consumption)} días")
print(f"Consumo medio diario: {daily_consumption['consumo_medio'].mean():.2f}")
print(f"Desviación estándar: {daily_consumption['consumo_medio'].std():.2f}")

# Visualizar la serie completa
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(daily_consumption.index, daily_consumption['consumo_medio'], color='#3498db', linewidth=1.2)
ax.set_title('Consumo medio diario de combustible - Flota completa', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Tasa de consumo media')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---

## 2. Forecasts Baseline

Antes de aplicar modelos sofisticados, implementamos tres baselines simples para tener un punto de referencia:

- **Naive:** La predicción para mañana es el valor de hoy.
- **Seasonal Naive:** La predicción para mañana es el valor del mismo día de la semana pasada.
- **Moving Average:** La predicción para mañana es el promedio de los últimos 7 días.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def rmse(y_true, y_pred):
    """Root Mean Squared Error."""
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Train/test split temporal (80/20)
n = len(daily_consumption)
train_size = int(n * 0.8)
train = daily_consumption.iloc[:train_size]
test = daily_consumption.iloc[train_size:]

print(f"Train: {len(train)} días ({train.index.min()} a {train.index.max()})")
print(f"Test:  {len(test)} días ({test.index.min()} a {test.index.max()})")

In [ ]:
# --- Baseline 1: Naive forecast ---
naive_pred = np.full(len(test), train['consumo_medio'].iloc[-1])

# --- Baseline 2: Seasonal naive (mismo día de la semana pasada) ---
seasonal_pred = []
full_series = daily_consumption['consumo_medio'].values
for i in range(train_size, n):
    # Valor de hace 7 días
    if i - 7 >= 0:
        seasonal_pred.append(full_series[i - 7])
    else:
        seasonal_pred.append(full_series[i - 1])
seasonal_pred = np.array(seasonal_pred)

# --- Baseline 3: Moving average (ventana de 7 días) ---
ma_pred = []
for i in range(train_size, n):
    window_start = max(0, i - 7)
    ma_pred.append(full_series[window_start:i].mean())
ma_pred = np.array(ma_pred)

y_test = test['consumo_medio'].values

print("=" * 55)
print(f"{'Método':<25} {'RMSE':>10} {'MAPE (%)':>10}")
print("=" * 55)
for name, pred in [('Naive', naive_pred), ('Seasonal Naive', seasonal_pred), ('Moving Average (7d)', ma_pred)]:
    r = rmse(y_test, pred)
    m = mape(y_test, pred)
    print(f"{name:<25} {r:>10.4f} {m:>10.2f}")
print("=" * 55)

In [ ]:
# Visualización de cada baseline vs actual
fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)

baselines = [
    ('Naive', naive_pred, '#e74c3c'),
    ('Seasonal Naive (7 días)', seasonal_pred, '#9b59b6'),
    ('Moving Average (7 días)', ma_pred, '#2ecc71')
]

for ax, (name, pred, color) in zip(axes, baselines):
    ax.plot(train.index, train['consumo_medio'], color='#3498db', alpha=0.5, label='Train')
    ax.plot(test.index, y_test, color='#3498db', linewidth=2, label='Actual (Test)')
    ax.plot(test.index, pred, color=color, linewidth=2, linestyle='--', label=f'Predicción: {name}')
    ax.axvline(x=test.index[0], color='gray', linestyle=':', alpha=0.7, label='Inicio test')
    ax.set_title(f'Baseline: {name} — MAPE: {mape(y_test, pred):.2f}%', fontsize=13, fontweight='bold')
    ax.set_ylabel('Consumo medio')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Fecha')
plt.tight_layout()
plt.show()

---

## 3. Modelo ARIMA

Ahora aplicamos ARIMA. Primero analizamos ACF y PACF para seleccionar los parámetros (p, d, q), y luego ajustamos el modelo.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Test de Dickey-Fuller para estacionariedad
result = adfuller(train['consumo_medio'].dropna())
print('Test de Dickey-Fuller Aumentado:')
print(f'  Estadístico ADF: {result[0]:.4f}')
print(f'  p-valor:         {result[1]:.4f}')
print(f'  Valores críticos:')
for key, value in result[4].items():
    print(f'    {key}: {value:.4f}')

if result[1] < 0.05:
    print('\n-> La serie ES estacionaria (p < 0.05). Sugerimos d=0.')
    d_suggested = 0
else:
    print('\n-> La serie NO es estacionaria (p >= 0.05). Sugerimos d=1.')
    d_suggested = 1

In [ ]:
# Gráficos ACF y PACF
series_to_plot = train['consumo_medio'].diff().dropna() if d_suggested == 1 else train['consumo_medio']
title_suffix = ' (diferenciada d=1)' if d_suggested == 1 else ' (original)'

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_acf(series_to_plot, lags=30, ax=axes[0], color='#3498db')
axes[0].set_title(f'ACF{title_suffix}', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rezago (lag)')
axes[0].set_ylabel('Autocorrelación')

plot_pacf(series_to_plot, lags=30, ax=axes[1], color='#e74c3c', method='ywm')
axes[1].set_title(f'PACF{title_suffix}', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Rezago (lag)')
axes[1].set_ylabel('Autocorrelación parcial')

plt.tight_layout()
plt.show()

print("\nInterpretación:")
print("- PACF: Los rezagos significativos sugieren el valor de p (componente AR).")
print("- ACF: Los rezagos significativos sugieren el valor de q (componente MA).")
print("- Observa dónde las barras caen dentro de la banda de confianza (zona azul claro).")

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# Intentar auto_arima si pmdarima está disponible
try:
    from pmdarima import auto_arima
    print("pmdarima disponible. Usando auto_arima para selección automática de parámetros...")
    auto_model = auto_arima(
        train['consumo_medio'],
        seasonal=False,
        stepwise=True,
        suppress_warnings=True,
        trace=True,
        max_p=5, max_q=5, max_d=2
    )
    best_order = auto_model.order
    print(f"\nMejor orden encontrado: ARIMA{best_order}")
    print(auto_model.summary())

except ImportError:
    print("pmdarima no disponible. Selección manual de parámetros basada en ACF/PACF.")
    # Heurística basada en ACF/PACF
    # PACF significativo en lag 1-2 -> p=2
    # ACF significativo en lag 1 -> q=1
    best_order = (2, d_suggested, 1)
    print(f"Orden seleccionado manualmente: ARIMA{best_order}")

# Ajustar modelo ARIMA
arima_model = ARIMA(train['consumo_medio'], order=best_order)
arima_result = arima_model.fit()
print("\n" + "=" * 50)
print("Resumen del modelo ARIMA:")
print("=" * 50)
print(arima_result.summary())

In [ ]:
# Forecast de los próximos 7 días (después del train set)
forecast_steps = min(7, len(test))
forecast = arima_result.get_forecast(steps=forecast_steps)
forecast_mean = forecast.predicted_mean
forecast_ci = forecast.conf_int(alpha=0.05)

# Visualizar forecast con intervalos de confianza
fig, ax = plt.subplots(figsize=(16, 6))

# Últimos 30 días del train
tail_train = train.iloc[-30:]
ax.plot(tail_train.index, tail_train['consumo_medio'], color='#3498db', linewidth=2, label='Datos históricos')

# Datos reales del test (primeros 7 días)
test_7 = test.iloc[:forecast_steps]
ax.plot(test_7.index, test_7['consumo_medio'], color='#2ecc71', linewidth=2, marker='o', label='Actual')

# Forecast
ax.plot(forecast_mean.index, forecast_mean.values, color='#e74c3c', linewidth=2, marker='s', label='Forecast ARIMA')
ax.fill_between(
    forecast_ci.index,
    forecast_ci.iloc[:, 0],
    forecast_ci.iloc[:, 1],
    color='#e74c3c', alpha=0.15, label='Intervalo de confianza 95%'
)

ax.axvline(x=test.index[0], color='gray', linestyle=':', alpha=0.7)
ax.set_title(f'Forecast ARIMA{best_order} - Próximos {forecast_steps} días', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Consumo medio diario')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

# Métricas del forecast
arima_rmse_7 = rmse(test_7['consumo_medio'].values, forecast_mean.values)
arima_mape_7 = mape(test_7['consumo_medio'].values, forecast_mean.values)
print(f"\nForecast ARIMA (7 días):")
print(f"  RMSE: {arima_rmse_7:.4f}")
print(f"  MAPE: {arima_mape_7:.2f}%")

---

## 4. Forecast por vehículo individual

Los patrones de consumo varían entre vehículos. Seleccionamos 3 vehículos de distintos tipos y comparamos sus pronósticos individuales con el de la flota.

In [ ]:
# Seleccionar 3 vehículos de diferentes tipos
vehicle_types_available = telemetry.groupby('vehicle_type')['vehicle_id'].nunique().reset_index()
print("Vehículos por tipo:")
print(vehicle_types_available)

selected_vehicles = []
for vtype in ['electrico', 'gasolina', 'hibrido']:
    candidates = telemetry[telemetry['vehicle_type'] == vtype]['vehicle_id'].unique()
    if len(candidates) > 0:
        # Elegir el vehículo con más registros
        counts = telemetry[telemetry['vehicle_id'].isin(candidates)].groupby('vehicle_id').size()
        selected_vehicles.append(counts.idxmax())

# Si no hay suficientes tipos, tomar los primeros 3 vehículos
if len(selected_vehicles) < 3:
    all_vehicles = telemetry['vehicle_id'].value_counts().head(3).index.tolist()
    selected_vehicles = all_vehicles[:3]

print(f"\nVehículos seleccionados: {selected_vehicles}")

In [ ]:
fig, axes = plt.subplots(len(selected_vehicles), 1, figsize=(16, 5 * len(selected_vehicles)), sharex=True)
if len(selected_vehicles) == 1:
    axes = [axes]

vehicle_forecasts = {}

for ax, vid in zip(axes, selected_vehicles):
    # Serie diaria del vehículo
    veh_daily = (
        telemetry[telemetry['vehicle_id'] == vid]
        .groupby('date')['fuel_consumption_rate']
        .mean()
    )
    veh_daily.index = pd.to_datetime(veh_daily.index)
    veh_daily = veh_daily.sort_index().asfreq('D')
    if veh_daily.isna().any():
        veh_daily = veh_daily.interpolate(method='linear')
    veh_daily = veh_daily.dropna()

    if len(veh_daily) < 15:
        ax.text(0.5, 0.5, f'{vid}: datos insuficientes ({len(veh_daily)} días)',
                transform=ax.transAxes, ha='center', va='center', fontsize=14)
        continue

    # Split y modelo
    vn = len(veh_daily)
    vtrain_size = int(vn * 0.8)
    vtrain = veh_daily.iloc[:vtrain_size]
    vtest = veh_daily.iloc[vtrain_size:]

    vtype = telemetry[telemetry['vehicle_id'] == vid]['vehicle_type'].iloc[0]
    color = vtype_colors.get(vtype, '#34495e')

    try:
        vmodel = ARIMA(vtrain, order=(1, d_suggested, 1))
        vresult = vmodel.fit()
        vforecast = vresult.get_forecast(steps=len(vtest))
        vpred = vforecast.predicted_mean
        vci = vforecast.conf_int(alpha=0.05)

        v_mape = mape(vtest.values, vpred.values)
        vehicle_forecasts[vid] = {'mape': v_mape, 'type': vtype}

        ax.plot(vtrain.index, vtrain.values, color=color, alpha=0.5, label='Train')
        ax.plot(vtest.index, vtest.values, color=color, linewidth=2, label='Actual')
        ax.plot(vpred.index, vpred.values, color='#e74c3c', linewidth=2, linestyle='--', label='Forecast')
        ax.fill_between(vci.index, vci.iloc[:, 0], vci.iloc[:, 1], color='#e74c3c', alpha=0.1)
        ax.axvline(x=vtest.index[0], color='gray', linestyle=':', alpha=0.5)
        ax.set_title(f'{vid} ({vtype}) — MAPE: {v_mape:.2f}%', fontsize=13, fontweight='bold')
    except Exception as e:
        ax.text(0.5, 0.5, f'{vid}: Error en ARIMA - {str(e)[:50]}',
                transform=ax.transAxes, ha='center', va='center', fontsize=12)

    ax.set_ylabel('Consumo medio')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Fecha')
plt.suptitle('Forecast individual por vehículo vs Flota', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

if vehicle_forecasts:
    print("\nComparación de MAPE individual vs flota:")
    print(f"  Flota completa (ARIMA): {arima_mape_7:.2f}%")
    for vid, info in vehicle_forecasts.items():
        print(f"  {vid} ({info['type']}): {info['mape']:.2f}%")

---

## 5. Evaluación comparativa de todos los métodos

Evaluamos todos los métodos en el período de test completo y comparamos su rendimiento.

In [ ]:
# Forecast ARIMA sobre todo el test set
arima_full_forecast = arima_result.get_forecast(steps=len(test))
arima_full_pred = arima_full_forecast.predicted_mean.values

# Tabla comparativa completa
results = {
    'Naive': {'pred': naive_pred, 'rmse': rmse(y_test, naive_pred), 'mape': mape(y_test, naive_pred)},
    'Seasonal Naive': {'pred': seasonal_pred, 'rmse': rmse(y_test, seasonal_pred), 'mape': mape(y_test, seasonal_pred)},
    'Moving Avg (7d)': {'pred': ma_pred, 'rmse': rmse(y_test, ma_pred), 'mape': mape(y_test, ma_pred)},
    f'ARIMA{best_order}': {'pred': arima_full_pred, 'rmse': rmse(y_test, arima_full_pred), 'mape': mape(y_test, arima_full_pred)},
}

results_df = pd.DataFrame({
    'Método': results.keys(),
    'RMSE': [v['rmse'] for v in results.values()],
    'MAPE (%)': [v['mape'] for v in results.values()]
}).sort_values('MAPE (%)')

print("\n" + "=" * 55)
print("COMPARACIÓN DE MÉTODOS DE FORECASTING")
print("=" * 55)
print(results_df.to_string(index=False))
print("=" * 55)

In [ ]:
# Gráfico de barras comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_bar = ['#3498db', '#9b59b6', '#2ecc71', '#e74c3c']
methods = list(results.keys())

# RMSE
rmse_values = [results[m]['rmse'] for m in methods]
bars1 = axes[0].bar(methods, rmse_values, color=colors_bar[:len(methods)], edgecolor='white', linewidth=1.5)
axes[0].set_title('RMSE por método', fontsize=13, fontweight='bold')
axes[0].set_ylabel('RMSE')
for bar, val in zip(bars1, rmse_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)
axes[0].tick_params(axis='x', rotation=15)

# MAPE
mape_values = [results[m]['mape'] for m in methods]
bars2 = axes[1].bar(methods, mape_values, color=colors_bar[:len(methods)], edgecolor='white', linewidth=1.5)
axes[1].set_title('MAPE (%) por método', fontsize=13, fontweight='bold')
axes[1].set_ylabel('MAPE (%)')
for bar, val in zip(bars2, mape_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.2f}%', ha='center', va='bottom', fontsize=10)
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Evaluación comparativa de métodos de forecasting', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Análisis de residuos del modelo ARIMA
residuals = arima_result.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Serie de residuos
axes[0, 0].plot(residuals.index, residuals.values, color='#3498db', linewidth=0.8)
axes[0, 0].axhline(y=0, color='red', linestyle='--', alpha=0.7)
axes[0, 0].set_title('Residuos a lo largo del tiempo', fontweight='bold')
axes[0, 0].set_xlabel('Fecha')
axes[0, 0].set_ylabel('Residuo')

# 2. Histograma de residuos
axes[0, 1].hist(residuals, bins=30, color='#3498db', edgecolor='white', density=True)
from scipy import stats
x_range = np.linspace(residuals.min(), residuals.max(), 100)
axes[0, 1].plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()),
               color='#e74c3c', linewidth=2, label='Normal teórica')
axes[0, 1].set_title('Distribución de residuos', fontweight='bold')
axes[0, 1].legend()

# 3. ACF de residuos
plot_acf(residuals, lags=20, ax=axes[1, 0], color='#3498db')
axes[1, 0].set_title('ACF de residuos', fontweight='bold')

# 4. QQ-plot
stats.probplot(residuals, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('QQ-Plot de residuos', fontweight='bold')
axes[1, 1].get_lines()[0].set_color('#3498db')
axes[1, 1].get_lines()[1].set_color('#e74c3c')

plt.suptitle(f'Diagnóstico de residuos - ARIMA{best_order}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Test de normalidad
stat_jb, p_jb = stats.jarque_bera(residuals)
print(f"Test de Jarque-Bera: estadístico={stat_jb:.4f}, p-valor={p_jb:.4f}")
if p_jb > 0.05:
    print("  -> Los residuos se distribuyen normalmente (buena señal).")
else:
    print("  -> Los residuos NO se distribuyen normalmente. El modelo podría mejorarse.")

---

## 6. Forecast de degradación de batería (vehículos eléctricos)

Para los vehículos eléctricos, es crítico monitorear la degradación de la batería. Pronosticamos la tendencia del `battery_soc_pct` para estimar cuándo la batería alcanzará el 80% de capacidad.

In [ ]:
# Filtrar vehículos eléctricos
electric_vehicles = telemetry[telemetry['vehicle_type'] == 'electrico']['vehicle_id'].unique()
print(f"Vehículos eléctricos: {len(electric_vehicles)}")

if len(electric_vehicles) > 0:
    # Consumo medio diario de SOC (State of Charge) por vehículo eléctrico
    ev_daily_soc = (
        telemetry[telemetry['vehicle_type'] == 'electrico']
        .groupby(['vehicle_id', 'date'])['battery_soc_pct']
        .mean()
        .reset_index()
    )
    ev_daily_soc['date'] = pd.to_datetime(ev_daily_soc['date'])

    # Tomar el SOC promedio diario de la flota eléctrica
    fleet_ev_soc = ev_daily_soc.groupby('date')['battery_soc_pct'].mean()
    fleet_ev_soc = fleet_ev_soc.sort_index().asfreq('D')
    if fleet_ev_soc.isna().any():
        fleet_ev_soc = fleet_ev_soc.interpolate(method='linear')

    print(f"SOC medio flota eléctrica: {fleet_ev_soc.mean():.2f}%")
    print(f"Tendencia (primer día vs último): {fleet_ev_soc.iloc[0]:.2f}% -> {fleet_ev_soc.iloc[-1]:.2f}%")
else:
    print("No se encontraron vehículos eléctricos. Se usará battery_soc_pct de toda la flota como ejemplo.")
    fleet_ev_soc = (
        telemetry.groupby('date')['battery_soc_pct'].mean()
    )
    fleet_ev_soc.index = pd.to_datetime(fleet_ev_soc.index)
    fleet_ev_soc = fleet_ev_soc.sort_index().asfreq('D')
    if fleet_ev_soc.isna().any():
        fleet_ev_soc = fleet_ev_soc.interpolate(method='linear')

In [ ]:
# Forecast de SOC con ARIMA
soc_train_size = int(len(fleet_ev_soc) * 0.8)
soc_train = fleet_ev_soc.iloc[:soc_train_size]

try:
    soc_model = ARIMA(soc_train, order=(1, 1, 1))
    soc_result = soc_model.fit()

    # Forecast a 30 días
    forecast_days = 30
    soc_forecast = soc_result.get_forecast(steps=forecast_days + len(fleet_ev_soc) - soc_train_size)
    soc_pred = soc_forecast.predicted_mean
    soc_ci = soc_forecast.conf_int(alpha=0.05)

    fig, ax = plt.subplots(figsize=(16, 6))

    ax.plot(fleet_ev_soc.index, fleet_ev_soc.values, color=vtype_colors['electrico'], linewidth=2, label='SOC actual')
    ax.plot(soc_pred.index, soc_pred.values, color='#e74c3c', linewidth=2, linestyle='--', label='Forecast SOC')
    ax.fill_between(soc_ci.index, soc_ci.iloc[:, 0], soc_ci.iloc[:, 1], color='#e74c3c', alpha=0.1)
    ax.axhline(y=80, color='orange', linestyle='--', linewidth=2, alpha=0.8, label='Umbral 80% capacidad')
    ax.axvline(x=fleet_ev_soc.index[-1], color='gray', linestyle=':', alpha=0.5, label='Último dato real')

    ax.set_title('Forecast de degradación de batería - Flota eléctrica', fontsize=14, fontweight='bold')
    ax.set_xlabel('Fecha')
    ax.set_ylabel('Battery SOC (%)')
    ax.legend(loc='lower left')
    ax.set_ylim(max(0, soc_ci.iloc[:, 0].min() - 5), min(105, soc_ci.iloc[:, 1].max() + 5))
    plt.tight_layout()
    plt.show()

    # Estimar cuándo llegaría a 80%
    below_80 = soc_pred[soc_pred < 80]
    if len(below_80) > 0:
        print(f"\nEl SOC medio caería por debajo del 80% alrededor del: {below_80.index[0].strftime('%Y-%m-%d')}")
        days_to_80 = (below_80.index[0] - fleet_ev_soc.index[-1]).days
        print(f"Eso es aproximadamente {days_to_80} días a partir del último dato.")
    else:
        print("\nEl SOC no se proyecta por debajo del 80% en el horizonte de forecast.")
        print("La batería mantiene buena capacidad según la tendencia actual.")

except Exception as e:
    print(f"Error en forecast de SOC: {e}")
    print("Se requiere más datos para un forecast confiable de degradación de batería.")

---

## 7. Resumen y conclusiones

### Hallazgos principales

1. **Comparación de métodos:** Evaluamos cuatro métodos de forecasting (Naive, Seasonal Naive, Moving Average y ARIMA) sobre el consumo diario de la flota. El mejor método según MAPE nos da la mayor precisión para proyecciones de costo.

2. **Patrones individuales vs flota:** Los vehículos individuales muestran patrones de consumo distintos al promedio de la flota. Esto sugiere que un modelo por vehículo (o por tipo de vehículo) podría ser más preciso que un modelo global.

3. **Degradación de batería:** El forecast de SOC para vehículos eléctricos permite planificar el reemplazo de baterías antes de que la capacidad caiga a niveles críticos.

4. **Implicaciones de costo:** Con un forecast confiable del consumo de combustible, podemos:
   - Proyectar costos operativos mensuales y trimestrales
   - Identificar tendencias de incremento de costo tempranamente
   - Optimizar la estrategia de compra de combustible/electricidad

### Limitaciones

- ARIMA asume linealidad; modelos como Prophet o LSTM podrían capturar patrones no lineales.
- No incluimos variables exógenas (clima, tipo de ruta, carga) que podrían mejorar las predicciones.
- La estacionalidad semanal podría modelarse explícitamente con SARIMA.

### Próximo notebook

En el [05 - Detección de Anomalías](05_anomaly_detection.ipynb) utilizaremos técnicas de detección de anomalías para identificar automáticamente comportamientos inusuales en la flota, complementando el forecasting con alertas tempranas.